<a href="https://colab.research.google.com/github/nyp-sit/iti121-2025s2/blob/main/L8/yolov11_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semantic and Instance Segmentation

## Overview

This notebook demonstrates how to use YOLOv11 for both **instance segmentation** and **semantic segmentation** tasks. YOLOv11 is the latest version of the YOLO (You Only Look Once) object detection family, which has been extended to support segmentation tasks.

## What You'll Learn

1. **Instance Segmentation**: Identifying and segmenting individual objects with precise pixel-level masks
2. **Semantic Segmentation**: Classifying every pixel in an image into predefined categories

## Key Concepts

### Instance Segmentation
- **Purpose**: Detect, classify, and segment individual object instances
- **Output**: Bounding boxes + pixel-level masks for each detected object
- **Use Cases**: Autonomous driving, medical imaging, robotics

### Semantic Segmentation  
- **Purpose**: Classify every pixel in an image into semantic categories
- **Output**: Dense pixel-wise classification map
- **Use Cases**: Scene understanding, image editing, augmented reality

## Requirements
- Python 3.8+
- PyTorch
- YOLOv11
- OpenCV
- Matplotlib
- PIL/Pillow


## Installation and Setup

First, let's install the required packages. In Google Colab, you can run these commands directly. For local environments, make sure you have Python 3.8+ installed.


In [ ]:
# Install required packages
!pip install ultralytics opencv-python matplotlib pillow

In [ ]:
# Import required libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from ultralytics import YOLO
import os
from pathlib import Path

# Set up matplotlib for better visualization
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


## Part 1: Instance Segmentation with YOLOv11

### What is Instance Segmentation?

Instance segmentation combines object detection with semantic segmentation. It not only identifies objects in an image but also provides precise pixel-level masks for each individual object instance.

**Key Features:**
- **Object Detection**: Identifies bounding boxes and class labels
- **Pixel-level Masks**: Provides exact shape boundaries for each object
- **Instance Separation**: Distinguishes between multiple objects of the same class

### YOLOv11 Segmentation Models

YOLOv11 offers several pre-trained models for segmentation:
- `yolo11n-seg.pt` - Nano (fastest, smallest)
- `yolo11s-seg.pt` - Small
- `yolo11m-seg.pt` - Medium
- `yolo11l-seg.pt` - Large
- `yolo11x-seg.pt` - Extra Large (most accurate)

Let's start by loading a segmentation model and testing it on sample images.


In [ ]:
# We will first download the test image
!wget -q https://ultralytics.com/images/bus.jpg -O sample_image.jpg

In [ ]:
# Load YOLOv11 segmentation model
# We'll use the medium model for a good balance of speed and accuracy
model = YOLO('yolo11m-seg.pt')

print("YOLOv11 segmentation model loaded successfully!")

sample_image_path = "sample_image.jpg"

In [ ]:
img = cv2.imread(sample_image_path)
img.shape

In [ ]:
# Perform instance segmentation
print("Running instance segmentation...")
results = model(sample_image_path)

# Display the original image
plt.figure(figsize=(10, 5))

# Original image
plt.subplot(1, 2, 1)
original_img = cv2.imread(sample_image_path)
original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
plt.imshow(original_img)
plt.title("Original Image")
plt.axis('off')

# Results with bounding boxes and masks
plt.subplot(1, 2, 2)
result_img = results[0].plot()
plt.imshow(result_img)
plt.title("Instance Segmentation Results")
plt.axis('off')



Let's extract and display the individual masks.

In [ ]:
# Extract and display individual masks
plt.subplot(1, 3, 3)

# make sure we have some masks to display
if len(results[0].masks) > 0:
    # Create a combined mask visualization
    mask_img = np.zeros_like(original_img)  # create a empty black canvas of the same size as original image
    colors = plt.cm.tab10(np.linspace(0, 1, len(results[0].masks))) # create a color map for each mask

    for i, mask in enumerate(results[0].masks.data):
        mask_np = mask.cpu().numpy()

        # by default, YOLO11 resize the image to (640,640) before going through the inference
        # so the output masks is also (640,640), so we need to rezie them to original image size
        # cv2 covention is (width, height), and torch convention is (height, width)
        mask_resized = cv2.resize(mask_np, (original_img.shape[1], original_img.shape[0]))
        mask_img[mask_resized >= 0.25] = colors[i][:3] * 255

    plt.imshow(mask_img.astype(np.uint8))
    plt.title(f"Individual Masks ({len(results[0].masks)} objects)")
else:
    plt.imshow(original_img)
    plt.title("No objects detected")
plt.axis('off')

plt.tight_layout()
plt.show()

# Print detection results
print(f"\nDetection Results:")
print(f"Number of objects detected: {len(results[0].boxes) if results[0].boxes is not None else 0}")
if results[0].boxes is not None:
    for i, box in enumerate(results[0].boxes):
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        class_name = model.names[class_id]
        print(f"Object {i+1}: {class_name} (confidence: {confidence:.2f})")

### Understanding Instance Segmentation Results

The code above demonstrates several key aspects of instance segmentation:

1. **Model Loading**: We load the YOLOv11 medium segmentation model which provides a good balance between speed and accuracy.

2. **Inference**: The `model(image_path)` call performs inference and returns results containing:
   - **Bounding boxes**: Coordinates and confidence scores
   - **Class predictions**: Object class labels
   - **Masks**: Pixel-level segmentation masks

   You can also call `model.predict()`.

3. **Visualization**: We display three views:
   - Original image
   - Results with bounding boxes and masks overlaid
   - Individual masks colored by object instance

### Key Parameters for Instance Segmentation

You can customize the segmentation behavior using various parameters:

- `conf`: Confidence threshold (default: 0.25)
- `iou`: Intersection over Union threshold for NMS (default: 0.7)
- `max_det`: Maximum number of detections (default: 300)
- `device`: Device to run inference on ('cpu', 'cuda', etc.)

You can find more parameters at the [official site](https://docs.ultralytics.com/modes/predict/#inference-arguments)

**Exercises**:

You can try changing some of the parameters to see what happens.

## Part 2: Semantic Segmentation with YOLOv11

### What is Semantic Segmentation?

Semantic segmentation assigns a class label to every pixel in an image, creating a dense pixel-wise classification map. Unlike instance segmentation, it doesn't distinguish between individual object instances of the same class.

**Key Differences from Instance Segmentation:**
- **Pixel-level Classification**: Every pixel gets a class label
- **No Instance Separation**: Multiple objects of the same class are treated as one region
- **Dense Output**: Produces a complete segmentation map covering the entire image

### Converting Instance Segmentation to Semantic Segmentation

While YOLOv11 is primarily designed for instance segmentation, we can convert its output to semantic segmentation by:
1. Combining masks from objects of the same class
2. Creating a unified segmentation map
3. Assigning class labels to each pixel


In [ ]:
# Function to convert instance segmentation to semantic segmentation
def instance_to_semantic_segmentation(results, image_shape):
    """
    Convert YOLO instance segmentation results to semantic segmentation

    Args:
        results: YOLO results object
        image_shape: (height, width) of the original image

    Returns:
        semantic_mask: 2D array with class IDs for each pixel
        class_colors: Dictionary mapping class IDs to colors
    """
    height, width = image_shape[:2]
    semantic_mask = np.zeros((height, width), dtype=np.int32)

    if results[0].masks is not None and results[0].boxes is not None:
        # Get unique class IDs
        class_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        masks = results[0].masks.data.cpu().numpy()

        # Process each detection
        for i, (class_id, mask) in enumerate(zip(class_ids, masks)):
            # Resize mask to original image size
            mask_resized = cv2.resize(mask, (width, height))

            # Set pixels where mask is active to the class ID, offset by 1
            # this is to differentiate those position that is background
            semantic_mask[mask_resized > 0.5] = class_id + 1

    return semantic_mask, results[0].names

# Perform semantic segmentation
print("Converting to semantic segmentation...")
results = model(sample_image_path)

# Get original image dimensions
original_img = cv2.imread(sample_image_path)
original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)

# Convert to semantic segmentation
semantic_mask, class_names = instance_to_semantic_segmentation(results, original_img.shape)

# Create visualization
plt.figure(figsize=(15, 5))

# Original image
plt.subplot(1, 3, 1)
plt.imshow(original_img)
plt.title("Original Image")
plt.axis('off')

# Instance segmentation
plt.subplot(1, 3, 2)
instance_result = results[0].plot()
plt.imshow(instance_result)
plt.title("Instance Segmentation")
plt.axis('off')

# Semantic segmentation
plt.subplot(1, 3, 3)
# Create colored semantic mask
unique_classes = np.unique(semantic_mask)
semantic_colored = np.zeros((*semantic_mask.shape, 3), dtype=np.uint8)

# Assign colors to each class
colors = plt.cm.Set3(np.linspace(0, 1, len(unique_classes)))
for i, class_id in enumerate(unique_classes):
    if class_id > 0:  # Skip background (class_id = 0)
        mask = semantic_mask == class_id

        # color the pixels corresponding to the class to the assigned color
        semantic_colored[mask] = colors[i][:3] * 255

plt.imshow(semantic_colored)
plt.title("Semantic Segmentation")
plt.axis('off')

plt.tight_layout()
plt.show()

# Print semantic segmentation statistics
print(f"\nSemantic Segmentation Results:")
print(f"Number of unique classes detected: {len(unique_classes) - 1}")  # -1 for background
for class_id in unique_classes:
    if class_id > 0:
        class_name = class_names[class_id-1]
        pixel_count = np.sum(semantic_mask == class_id)
        percentage = (pixel_count / semantic_mask.size) * 100
        print(f"Class {class_id} ({class_name}): {pixel_count} pixels ({percentage:.1f}%)")
